In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

/var/folders/nk/7z1ywx_15gq_crc3996dd4vm0000gn/T/ipykernel_49174/1978443558.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [ ]:
df = pd.read_csv("data/kyiv_flats_data.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10362 entries, 0 to 10361
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   url                10362 non-null  object 
 1   price              10308 non-null  float64
 2   currency           10308 non-null  object 
 3   address            10016 non-null  object 
 4   rooms              10307 non-null  float64
 5   area_total         10308 non-null  float64
 6   area_living        10255 non-null  object 
 7   area_kitchen       10255 non-null  object 
 8   floor              10308 non-null  object 
 9   total_floors       10304 non-null  float64
 10  build_year         8358 non-null   float64
 11  construction_tech  9438 non-null   object 
 12  heating_type       9547 non-null   object 
 13  lat                10025 non-null  float64
 14  lon                10025 non-null  float64
 15  district           8923 non-null   object 
 16  commission         348

In [1]:

# # Відбираємо лише числові колонки для кореляції
# numeric_df = df.select_dtypes(include=['float64', 'int64'])

# # Будуємо теплову карту (heatmap)
# plt.figure(figsize=(10, 8))
# sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
# plt.title('Матриця кореляцій')
# plt.show()


# Chapter 1: Feature Engineering

We enrich the dataset with geospatial features derived from each flat's coordinates (`lat`, `lon`).

**Features to add:**
| Feature | Description |
|---|---|
| `dist_metro_km` | Distance to nearest Kyiv metro station |
| `drive_time_maidan_car_min` | Drive time to Maidan Nezalezhnosti by **car** (Google API) |
| `drive_time_maidan_transit_min` | Travel time to Maidan by **public transport** (Google API) |
| `dist_bus_stop_km` | Distance to nearest bus/tram/trolley stop (OSM) |
| `bank_side` | Left or right bank of the Dnipro river |
| `dist_dnipro_km` | Distance to the nearest point on the Dnipro river (OSM) |
| `dist_grocery_km` | Distance to nearest grocery / supermarket (OSM) |
| `dist_school_km` | Distance to nearest school (OSM) |
| `dist_park_km` | Distance to nearest park (OSM) |

In [ ]:
import os
import json
import pickle
import time
import numpy as np
import requests
from math import radians
from scipy.spatial import cKDTree
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

# Work only on rows that have coordinates
df_geo = df.dropna(subset=["lat", "lon"]).copy()
print(f"Rows with coordinates: {len(df_geo)} / {len(df)}")

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in kilometres."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def nearest_dist_km(flat_lats, flat_lons, poi_lats, poi_lons):
    """Return the distance (km) from each flat to the nearest point in a POI set.
    Uses a KD-tree on (lat, lon) in radians — fast for large POI lists.
    """
    flat_rad = np.column_stack([np.radians(flat_lats), np.radians(flat_lons)])
    poi_rad = np.column_stack([np.radians(poi_lats), np.radians(poi_lons)])
    tree = cKDTree(poi_rad)
    chord_dist, _ = tree.query(flat_rad)
    # chord distance → km via R * 2 * arcsin(chord/2)
    return 6371.0 * 2 * np.arcsin(np.clip(chord_dist / 2, 0, 1))

## 1.1 Distance to Nearest Metro Station

Uses the local `metro_stations.json` file — no API calls needed.

In [ ]:
with open("data/metro_stations.json") as f:
    metro_data = json.load(f)

metro_stations = [
    station
    for line in metro_data["kyiv_metro"]["lines"]
    for station in line["stations"]
]

metro_lats = np.array([s["coordinates"]["lat"] for s in metro_stations])
metro_lons = np.array([s["coordinates"]["lon"] for s in metro_stations])

df_geo["dist_metro_km"] = nearest_dist_km(
    df_geo["lat"].values, df_geo["lon"].values, metro_lats, metro_lons
)

print(df_geo["dist_metro_km"].describe().round(3))

## 1.2 Travel Time to Maidan Nezalezhnosti (Google Distance Matrix API)

Fetches both **driving** and **public transit** travel times. Results are cached per mode in separate `.pkl` files so re-running the notebook does not re-bill the API. Batch size is 25 origins per request (Google's limit).

> **API key needed:** Enable the **Distance Matrix API** at `console.cloud.google.com`. Add `GOOGLE_MAPS_API_KEY=<your_key>` to a `.env` file in the project root.

In [ ]:
MAIDAN_LAT, MAIDAN_LON = 50.4501, 30.5234

# --- API test: single request from Khreshchatyk to Maidan ---
_test_origin = "50.4447,30.5228"  # Khreshchatyk metro
_test_dest   = f"{MAIDAN_LAT},{MAIDAN_LON}"

_resp = requests.get(
    "https://maps.googleapis.com/maps/api/distancematrix/json",
    params={"origins": _test_origin, "destinations": _test_dest,
            "mode": "driving", "key": GOOGLE_API_KEY}
).json()

print("Status:", _resp.get("status"))
print("Error message:", _resp.get("error_message", "—"))

if _resp.get("rows"):
    _el = _resp["rows"][0]["elements"][0]
    if _el["status"] == "OK":
        print(f"\nAPI works! Khreshchatyk → Maidan by car: {_el['duration']['value']//60} min, {_el['distance']['value']/1000:.2f} km")
    else:
        print(f"\nRow-level error — element status: {_el['status']}")

In [ ]:
MAIDAN_LAT, MAIDAN_LON = 50.4501, 30.5234
BATCH_SIZE = 25


def fetch_travel_times(origins, destination, api_key, mode, batch_size=BATCH_SIZE):
    """Call Google Distance Matrix API in batches for a given mode.

    mode: 'driving' or 'transit'
    Returns a list of travel times in minutes (np.nan where unavailable).
    """
    cache_file = f"cache/drive_time_cache_{mode}.pkl"
    cache = {}
    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            cache = pickle.load(f)

    uncached = [o for o in origins if o not in cache]

    for i in range(0, len(uncached), batch_size):
        batch = uncached[i : i + batch_size]
        origins_str = "|".join(f"{lat},{lon}" for lat, lon in batch)
        dest_str = f"{destination[0]},{destination[1]}"
        url = (
            "https://maps.googleapis.com/maps/api/distancematrix/json"
            f"?origins={origins_str}&destinations={dest_str}"
            f"&mode={mode}&key={api_key}"
        )
        resp = requests.get(url).json()
        for j, row in enumerate(resp.get("rows", [])):
            element = row["elements"][0]
            if element["status"] == "OK":
                cache[batch[j]] = element["duration"]["value"] / 60.0
            else:
                cache[batch[j]] = np.nan
        time.sleep(0.1)

    with open(cache_file, "wb") as f:
        pickle.dump(cache, f)

    return [cache.get(o, np.nan) for o in origins]


if GOOGLE_API_KEY:
    origins = list(zip(df_geo["lat"].values, df_geo["lon"].values))

    car_times = fetch_travel_times(origins, (MAIDAN_LAT, MAIDAN_LON), GOOGLE_API_KEY, mode="driving")
    df_geo["drive_time_maidan_car_min"] = pd.to_numeric(car_times, errors="coerce")

    transit_times = fetch_travel_times(origins, (MAIDAN_LAT, MAIDAN_LON), GOOGLE_API_KEY, mode="transit")
    df_geo["drive_time_maidan_transit_min"] = pd.to_numeric(transit_times, errors="coerce")

    print("Car:")
    print(df_geo["drive_time_maidan_car_min"].describe().round(1))
    print("\nPublic transit:")
    print(df_geo["drive_time_maidan_transit_min"].describe().round(1))
else:
    print("GOOGLE_MAPS_API_KEY not set — skipping travel time features.")

## 1.3 Distance to Nearest Bus / Tram / Trolleybus Stop (OpenStreetMap)

Uses the free Overpass API — no key required. Fetches all public transit stops in Kyiv once, then computes nearest-neighbour distances locally.

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
OVERPASS_HEADERS = {"User-Agent": "kyiv-flats-ml/1.0 (research project)"}


def overpass_query(query, cache_file, retries=3):
    """Run an Overpass QL query, cache result as pickle. Tries multiple endpoints."""
    cache_path = f"cache/{cache_file}"
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    last_error = None
    for endpoint in OVERPASS_ENDPOINTS:
        for attempt in range(retries):
            try:
                resp = requests.get(
                    endpoint,
                    params={"data": query},
                    headers=OVERPASS_HEADERS,
                    timeout=120,
                )
                resp.raise_for_status()
                data = resp.json()
                with open(cache_path, "wb") as f:
                    pickle.dump(data, f)
                print(f"  OK ({endpoint})")
                return data
            except Exception as e:
                last_error = e
                wait = 5 * (attempt + 1)
                print(f"  {endpoint} attempt {attempt+1} failed: {e} — retrying in {wait}s")
                time.sleep(wait)

    raise RuntimeError(f"All Overpass endpoints failed. Last error: {last_error}")


def extract_coords(overpass_data):
    """Extract (lat, lon) from Overpass nodes/ways/relations."""
    lats, lons = [], []
    for el in overpass_data.get("elements", []):
        if "lat" in el and "lon" in el:
            lats.append(el["lat"])
            lons.append(el["lon"])
        elif "center" in el:
            lats.append(el["center"]["lat"])
            lons.append(el["center"]["lon"])
    return np.array(lats), np.array(lons)


# Kyiv bounding box: south, west, north, east
KYIV_BB = "50.2,30.2,50.6,30.9"

bus_query = f"""
[out:json][timeout:90];
node["highway"="bus_stop"]({KYIV_BB});
out body;
"""
bus_data = overpass_query(bus_query, "cache_bus_stops.pkl")
bus_lats, bus_lons = extract_coords(bus_data)
print(f"Bus/tram stops fetched: {len(bus_lats)}")

df_geo["dist_bus_stop_km"] = nearest_dist_km(
    df_geo["lat"].values, df_geo["lon"].values, bus_lats, bus_lons
)
print(df_geo["dist_bus_stop_km"].describe().round(3))

## 1.4 Left / Right Bank of the Dnipro

Uses the actual Dnipro river polygon from `dnipro_kyiv_simplified.geojson`.
For each flat a ray is cast eastward — if it hits the river polygon the flat is **west of the river → right bank**; if it misses the flat is **east of the river → left bank**.

In [ ]:
import json
from shapely.geometry import Point, LineString, shape
from shapely.ops import nearest_points, unary_union

# Load Dnipro polygon (shared by bank_side and dist_dnipro)
with open("data/dnipro_kyiv_simplified.geojson") as f:
    dnipro_geojson = json.load(f)

dnipro_geom = unary_union([
    shape(feat["geometry"])
    for feat in dnipro_geojson["features"]
])
print(f"Dnipro geometry type: {dnipro_geom.geom_type}, bounds: {[round(x,4) for x in dnipro_geom.bounds]}")


def get_bank_side(lon, lat, river_geom):
    """Right bank = west of Dnipro, left bank = east of Dnipro.
    Cast a ray eastward: if it intersects the river, the flat is west of it (right bank).
    """
    pt = Point(lon, lat)
    if pt.within(river_geom):
        return "river"
    ray = LineString([(lon, lat), (lon + 5, lat)])
    if not ray.intersection(river_geom).is_empty:
        return "right"
    return "left"


df_geo["bank_side"] = df_geo.apply(
    lambda row: get_bank_side(row["lon"], row["lat"], dnipro_geom), axis=1
)
print(df_geo["bank_side"].value_counts())

## 1.5 Distance from the Dnipro River

Uses the same `dnipro_kyiv_simplified.geojson` polygon. For each flat, finds the nearest point on the river boundary and computes the haversine distance in km. Flats inside the polygon (on the river) get distance 0.

In [ ]:
def dist_to_river_km(lon, lat, river_geom):
    """Haversine distance (km) from a point to the nearest point on the river boundary.
    Returns 0 if the point is inside the river polygon.
    """
    pt = Point(lon, lat)
    if pt.within(river_geom):
        return 0.0
    nearest_pt = nearest_points(pt, river_geom.boundary)[1]
    return haversine_km(lat, lon, nearest_pt.y, nearest_pt.x)


df_geo["dist_dnipro_km"] = df_geo.apply(
    lambda row: dist_to_river_km(row["lon"], row["lat"], dnipro_geom), axis=1
)
print(df_geo["dist_dnipro_km"].describe().round(3))

## 1.6 Distance to Nearest Grocery Store

Two features:
- `dist_grocery_km` — nearest any grocery / convenience store (OSM)
- `dist_supermarket_km` — nearest major chain supermarket (Сільпо, АТБ, Варус, Новус, Ашан, Велмарт, Thrash!)

In [ ]:
grocery_query = f"""
[out:json][timeout:90];
(
  node["shop"~"supermarket|convenience|grocery"]({KYIV_BB});
  way["shop"~"supermarket|convenience|grocery"]({KYIV_BB});
);
out center;
"""
grocery_data = overpass_query(grocery_query, "cache_grocery.pkl")
grocery_lats, grocery_lons = extract_coords(grocery_data)
print(f"Grocery stores fetched: {len(grocery_lats)}")

df_geo["dist_grocery_km"] = nearest_dist_km(
    df_geo["lat"].values, df_geo["lon"].values, grocery_lats, grocery_lons
)
print(df_geo["dist_grocery_km"].describe().round(3))

In [ ]:
# Major supermarket chains present in Kyiv
SUPERMARKET_CHAINS = "Сільпо|Silpo|АТБ|ATB|Варус|Varus|Новус|Novus|Ашан|Ashan|Auchan|Велмарт|Velmart|Thrash"

supermarket_query = f"""
[out:json][timeout:90];
(
  node["shop"="supermarket"]["name"~"{SUPERMARKET_CHAINS}",i]({KYIV_BB});
  way["shop"="supermarket"]["name"~"{SUPERMARKET_CHAINS}",i]({KYIV_BB});
);
out center;
"""
supermarket_data = overpass_query(supermarket_query, "cache_supermarkets.pkl")
supermarket_lats, supermarket_lons = extract_coords(supermarket_data)
print(f"Major supermarkets fetched: {len(supermarket_lats)}")

df_geo["dist_supermarket_km"] = nearest_dist_km(
    df_geo["lat"].values, df_geo["lon"].values, supermarket_lats, supermarket_lons
)
print(df_geo["dist_supermarket_km"].describe().round(3))

## 1.7 Distance to Nearest School

In [ ]:
school_query = f"""
[out:json][timeout:90];
(
  node["amenity"="school"]({KYIV_BB});
  way["amenity"="school"]({KYIV_BB});
);
out center;
"""
school_data = overpass_query(school_query, "cache_schools.pkl")
school_lats, school_lons = extract_coords(school_data)
print(f"Schools fetched: {len(school_lats)}")

df_geo["dist_school_km"] = nearest_dist_km(
    df_geo["lat"].values, df_geo["lon"].values, school_lats, school_lons
)
print(df_geo["dist_school_km"].describe().round(3))

## 1.8 Distance to Nearest Park

In [ ]:
park_query = f"""
[out:json][timeout:90];
(
  node["leisure"="park"]({KYIV_BB});
  way["leisure"="park"]({KYIV_BB});
);
out center;
"""
park_data = overpass_query(park_query, "cache_parks.pkl")
park_lats, park_lons = extract_coords(park_data)
print(f"Parks fetched: {len(park_lats)}")

df_geo["dist_park_km"] = nearest_dist_km(
    df_geo["lat"].values, df_geo["lon"].values, park_lats, park_lons
)
print(df_geo["dist_park_km"].describe().round(3))

## 1.9 Save Enriched Dataset

Merge new features back into the full dataframe and save to `kyiv_flats_enriched.csv`.

In [ ]:
new_feature_cols = [
    "dist_metro_km",
    "drive_time_maidan_car_min",
    "drive_time_maidan_transit_min",
    "dist_bus_stop_km",
    "bank_side",
    "dist_dnipro_km",
    "dist_grocery_km",
    "dist_supermarket_km",
    "dist_school_km",
    "dist_park_km",
]
existing_cols = [c for c in new_feature_cols if c in df_geo.columns]

df_enriched = df.merge(
    df_geo[["url"] + existing_cols],
    on="url",
    how="left",
)

df_enriched.to_csv("data/kyiv_flats_enriched.csv", index=False)
print(f"Saved data/kyiv_flats_enriched.csv — shape: {df_enriched.shape}")
df_enriched[existing_cols].describe().round(3)